一年为单位，批量提取每个月的气象数据，转化为excel或者cvs。

In [1]:
#读取单个文件信息并转化为DataFrame
import xarray as xr
import pandas as pd
def Read_File(path):
    # 贵阳市经纬度范围为东经106° 07′ ～ 107°17 ′，纬度范围为北纬26°11′～ 27°22′（106.12°E~107.28°E，26.18°N~27.37°N）
    guiyang_lon_range = (106.12, 107.28)
    guiyang_lat_range = (26.18, 27.37)
    #读取气象数据
    ds=xr.open_dataset(path)
    ds_guiyang = ds.sel(lon=slice(*guiyang_lon_range),lat=slice(*guiyang_lat_range))    #筛选贵阳市范围内的格点（按经纬度切片）
    df_guiyang=ds_guiyang.to_dataframe().reset_index()    #转化为DataFrame
    print(f"转化为DataFrame数据后数据形状为:{df_guiyang.shape}")
    return df_guiyang

In [2]:
#绘制气象数据散点图以及曲面图
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
def DisplayData(ClimData,SaveImagePath,Date):
    #整理数据
    lat=ClimData.iloc[:,0].values
    lon=ClimData.iloc[:,1].values
    info=ClimData.iloc[:,2].values
    # 生成规则网格（用于3D曲面/网格图）
    lon_grid,lat_grid=np.meshgrid(np.linspace(lon.min(), lon.max(),116),np.linspace(lat.min(), lat.max(),119))
    pres_grid = griddata(points=(lon, lat),values=info, xi=(lon_grid, lat_grid), method="linear")    #数据插值
    #绘制3D散点
    fig1 = plt.figure(figsize=(10, 8))
    ax1 = fig1.add_subplot(111, projection="3d")
    ax1.set_title("3D scatter plot of grid-point atmospheric PET in Guiyang")
    scatter=ax1.scatter(lon,lat,info,c=info,cmap="viridis",s=10,alpha=0.8,edgecolors="k",linewidths=0.2)
    ax1.set_xlabel("lon/°E", fontsize=12, labelpad=10)
    ax1.set_ylabel("lat/°N", fontsize=12, labelpad=10)
    ax1.set_zlabel("PET/mm", fontsize=12, labelpad=10)
    cbar1 = fig1.colorbar(scatter, ax=ax1, shrink=0.6, pad=0.1)
    cbar1.set_label("PET/mm", fontsize=10)
    ax1.view_init(elev=30, azim=45)
    plt.tight_layout()
    plt.savefig(SaveImagePath+"/3d_scatter_plot.png", dpi=300, bbox_inches="tight")
    plt.close(fig1)
    #绘制3D曲面图
    fig2 = plt.figure(figsize=(10, 8))
    ax2 = fig2.add_subplot(111, projection="3d")
    norm = plt.Normalize(vmin=pres_grid.min(), vmax=pres_grid.max())
    cmap = plt.cm.viridis
    ax2.set_title("3D surface plot of grid-point atmospheric PET in Guiyang", fontsize=14, pad=20)
    surf=ax2.plot_surface(lon_grid,lat_grid,pres_grid,cmap=cmap,norm=norm,rstride=1,cstride=1,alpha=0.9,shade=False )
    ax2.plot_wireframe(lon_grid, lat_grid, pres_grid,color="k", linewidth=0.2, alpha=0.3)
    ax2.set_xlabel("lon/°E", fontsize=12, labelpad=10)
    ax2.set_ylabel("lat/°N", fontsize=12, labelpad=10)
    ax2.set_zlabel("PET/mm", fontsize=12, labelpad=10)
    cbar2 = fig2.colorbar(surf,ax=ax2, shrink=0.6, pad=0.1)
    cbar2.set_label("PET/mm", fontsize=10)
    ax2.view_init(elev=30, azim=45)
    plt.tight_layout()
    plt.savefig(SaveImagePath+"/3d_suface_plot"+Date+".png", dpi=300, bbox_inches="tight")
    plt.close(fig2)

In [3]:
#转化数据
import seaborn as sns
from scipy.stats import normaltest
def ToGetData(Data,SaveImagePath,Date):
    #绘制数据分布图
    plt.figure(figsize=(10, 8))
    sns.histplot(Data,kde=True)
    plt.savefig(SaveImagePath+'/distribution_plot'+Date+'.png',dpi=300, bbox_inches="tight")
    plt.close()
    #对数据进行正态检验
    stats,pavelue=normaltest(Data)
    if pavelue>=0.05:    #满足正态分布数据计算均值
        InfoData=np.mean(Data)
    else :    #不满足正态分布时计算中位数
        InfoData=np.median(Data)
    return stats,pavelue,InfoData

In [4]:
#提取时间信息
def GetDate(path):
    year=path.split('_')[-2]   #提取年信息
    month=path.split('_')[-1].split('.')[0]    #提取月信息
    date=str(year)+'-'+str(month)+'-01'
    return date

In [5]:
#采用多级文件读取方式批量处理
import os
def get_pressure_file_paths(root_dir):
    # 存储最终文件路径
    file_paths = []
    if not os.path.isdir(root_dir):   # 校验一级目录是否存在
        print(f"错误：一级目录 {root_dir} 不存在！")
        return file_paths
    # 遍历二级目录（年份文件夹）
    for year_dir_name in os.listdir(root_dir):
        # 拼接二级目录完整路径
        year_dir = os.path.join(root_dir, year_dir_name)
        #排除.ipynb_checkpoints
        if not os.path.isdir(year_dir) or year_dir_name == ".ipynb_checkpoints":
            continue
        # 遍历三级目录
        for filename in os.listdir(year_dir):
            # 拼接三级文件完整路径
            file_path = os.path.join(year_dir, filename)
            # 排除子文件夹
            if os.path.isfile(file_path) and filename != ".ipynb_checkpoints":
                file_paths.append(file_path)
    return file_paths
ROOT_DIR = "/root/ClimData/petPM"
#获取所有文件路径
pressure_files = get_pressure_file_paths(ROOT_DIR)
print(f"共找到 {len(pressure_files)} 个气象文件：")

共找到 60 个气象文件：


In [6]:
#进行批量化处理
import tqdm
from pandas import DataFrame
OutPutImage='/root/ToResult/ToProcessImage/petPM'    #定义输出文件夹
#数据整合
DF_date,DF_stats,DF_pvalue,DF_Clim=[],[],[],[]
for file_name in tqdm.tqdm(pressure_files):
    datepath=file_name.split('/')[-1]
    date=GetDate(path=datepath)    #提取时间信息
    DF_date.append(date)
    #提取贵阳市气象数据
    DataFrame_Guiyang=Read_File(path=file_name)
    #绘制数据分布图
    DisplayData(ClimData=DataFrame_Guiyang,SaveImagePath=OutPutImage,Date=date)
    #整合数据
    clim=DataFrame_Guiyang.iloc[:,-1]
    Stats,Pavelue,InfoData=ToGetData(Data=clim,SaveImagePath=OutPutImage,Date=date)
    DF_stats.append(Stats)
    DF_pvalue.append(Pavelue)
    DF_Clim.append(InfoData)
ToClimData=DataFrame()
ToClimData['Date']=DF_date
ToClimData['Stat']=DF_stats
ToClimData['P value']=DF_pvalue
ToClimData['petPM']=DF_Clim

  0%|          | 0/60 [00:00<?, ?it/s]

转化为DataFrame数据后数据形状为:(13804, 3)


  2%|▏         | 1/60 [00:06<06:37,  6.73s/it]

转化为DataFrame数据后数据形状为:(13804, 3)


  3%|▎         | 2/60 [00:11<05:15,  5.43s/it]

...

 97%|█████████▋| 58/60 [04:59<00:09,  4.72s/it]

转化为DataFrame数据后数据形状为:(13804, 3)


 98%|█████████▊| 59/60 [05:05<00:05,  5.26s/it]

转化为DataFrame数据后数据形状为:(13804, 3)


100%|██████████| 60/60 [05:10<00:00,  5.18s/it]


In [7]:
#输出为excel文件
Date=pd.to_datetime(ToClimData['Date'], errors="coerce")
ToClimData['Date']=Date
ToClimData=ToClimData.sort_values(by=ToClimData.columns[0],ascending=True,ignore_index=True)
ToClimData.to_csv('ToResult/ToProcessData/petPM.csv',encoding='utf-8')